# Results Visualization

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 140
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

ROOT = Path.cwd()
RESULT_DIR = ROOT / 'result'

FRAMEWORKS = ['fedavg', 'fedprox', 'fedditto']
FRAMEWORK_TITLES = {
    'fedavg': 'FedAvg',
    'fedprox': 'FedProx',
    'fedditto': 'FedDitto',
}
SETTINGS = [
    ('mnist', 'iid'),
    ('mnist', 'shard'),
    ('cifar10', 'iid'),
    ('cifar10', 'shard'),
]
SETTING_LABELS = {
    ('mnist', 'iid'): 'MNIST-IID',
    ('mnist', 'shard'): 'MNIST-NonIID',
    ('cifar10', 'iid'): 'CIFAR10-IID',
    ('cifar10', 'shard'): 'CIFAR10-NonIID',
}
SETTING_COLORS = {
    ('mnist', 'iid'): '#1f77b4',
    ('mnist', 'shard'): '#ff7f0e',
    ('cifar10', 'iid'): '#2ca02c',
    ('cifar10', 'shard'): '#d62728',
}

PRIMARY_METRIC_STYLE = {
    'test_accuracy': {'linestyle': '-', 'linewidth': 2.2, 'alpha': 0.95},
    'val_loss': {'linestyle': '-', 'linewidth': 2.2, 'alpha': 0.95},
}
AUC_STYLE = {'linestyle': '--', 'linewidth': 2.0, 'alpha': 0.9}

BASELINE_METHODS = ['local', 'fedavg', 'fedprox', 'fedditto']
BASELINE_TITLES = {
    'local': 'Local',
    'fedavg': 'FedAvg',
    'fedprox': 'FedProx',
    'fedditto': 'FedDitto',
}


In [ ]:
def load_mia_jsonl(path: Path) -> pd.DataFrame:
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if not rows:
        return pd.DataFrame()

    config = rows[0]
    metric_rows = rows[1:]
    mia_rows = [r for r in metric_rows if 'mia_round' in r]
    train_rows = [r for r in metric_rows if 'round' in r]

    mia_df = pd.DataFrame(mia_rows)
    train_df = pd.DataFrame(train_rows)

    if not mia_df.empty:
        mia_df = mia_df.rename(columns={'mia_round': 'round', 'auc': 'mia_auc'})
        mia_df = mia_df[['round', 'mia_auc']].copy()
    else:
        mia_df = pd.DataFrame(columns=['round', 'mia_auc'])

    if train_df.empty:
        merged = mia_df.copy()
    else:
        value_cols = [c for c in ['round', 'val_loss', 'test_accuracy', 'global_test_accuracy', 'personalized_val_loss'] if c in train_df.columns]
        train_df = train_df[value_cols].copy()
        if 'global_test_accuracy' in train_df.columns and 'test_accuracy' not in train_df.columns:
            train_df = train_df.rename(columns={'global_test_accuracy': 'test_accuracy'})
        if 'personalized_val_loss' in train_df.columns and 'val_loss' not in train_df.columns:
            train_df = train_df.rename(columns={'personalized_val_loss': 'val_loss'})
        merged = pd.merge(train_df, mia_df, on='round', how='outer').sort_values('round').reset_index(drop=True)

    parent_name = path.parent.name
    if parent_name.endswith('_dp_MIA'):
        framework = parent_name.replace('_dp_MIA', '')
        experiment_type = 'dp_mia'
    elif parent_name.endswith('_MIA'):
        framework = parent_name.replace('_MIA', '')
        experiment_type = 'mia'
    else:
        framework = parent_name
        experiment_type = 'unknown'

    merged['framework'] = framework
    merged['experiment_type'] = experiment_type
    merged['dataset'] = config.get('dataset')
    merged['partition'] = config.get('partition')
    merged['setting_label'] = SETTING_LABELS.get((merged['dataset'].iloc[0], merged['partition'].iloc[0]), f"{merged['dataset'].iloc[0]}-{merged['partition'].iloc[0]}")
    merged['source_file'] = str(path)
    return merged


def collect_all_results(result_dir: Path) -> pd.DataFrame:
    frames = []
    for framework in FRAMEWORKS:
        framework_dir = result_dir / f'{framework}_MIA'
        if not framework_dir.exists():
            continue
        for dataset, partition in SETTINGS:
            file_path = framework_dir / f'{dataset}_{partition}.jsonl'
            if file_path.exists():
                frames.append(load_mia_jsonl(file_path))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df = collect_all_results(RESULT_DIR)
df.head()


In [ ]:
if df.empty:
    print('No *_MIA jsonl files found under result/.')
else:
    summary = (
        df.groupby(['framework', 'dataset', 'partition'])
        .agg(
            rounds=('round', 'count'),
            has_acc=('test_accuracy', lambda s: int(s.notna().any())),
            has_val_loss=('val_loss', lambda s: int(s.notna().any())),
            has_mia_auc=('mia_auc', lambda s: int(s.notna().any())),
        )
        .reset_index()
        .sort_values(['framework', 'dataset', 'partition'])
    )
    display(summary)

In [ ]:
def add_combined_legend(fig, primary_name: str):
    color_handles = [
        Line2D([0], [0], color=SETTING_COLORS[key], lw=3, label=SETTING_LABELS[key])
        for key in SETTINGS
    ]
    metric_handles = [
        Line2D([0], [0], color='black', lw=2.4, linestyle='-', label=primary_name),
        Line2D([0], [0], color='black', lw=2.0, linestyle='--', label='MIA AUC'),
    ]
    fig.legend(handles=color_handles + metric_handles, loc='upper center', ncol=6, frameon=True, bbox_to_anchor=(0.5, 1.03))


def prepare_auc_curve(s: pd.DataFrame):
    if s.empty or 'round' not in s.columns:
        return pd.Series(dtype=float), pd.Series(dtype=float), pd.Series(dtype=float)
    rounds = pd.Index(range(int(s['round'].min()), int(s['round'].max()) + 1), name='round')
    auc_df = s[['round', 'mia_auc']].drop_duplicates('round').set_index('round').reindex(rounds)
    observed = auc_df['mia_auc'].copy()
    interpolated = auc_df['mia_auc'].interpolate(method='linear', limit_direction='both')
    return pd.Series(rounds, index=rounds), interpolated, observed


def plot_metric_vs_auc(df: pd.DataFrame, primary_metric: str, primary_label: str, title: str):
    fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharex=False)
    right_axes = []

    for ax, framework in zip(axes, FRAMEWORKS):
        ax2 = ax.twinx()
        right_axes.append(ax2)
        sub = df[df['framework'] == framework].copy()

        if sub.empty:
            ax.text(0.5, 0.5, 'No data yet', ha='center', va='center', fontsize=14, color='gray', transform=ax.transAxes)
            ax.set_title(FRAMEWORK_TITLES[framework], fontsize=14, weight='bold')
            ax.set_axis_off()
            ax2.set_axis_off()
            continue

        drawn_any = False
        for setting in SETTINGS:
            dataset, partition = setting
            s = sub[(sub['dataset'] == dataset) & (sub['partition'] == partition)].sort_values('round')
            if s.empty:
                continue

            color = SETTING_COLORS[setting]
            if primary_metric in s.columns and s[primary_metric].notna().any():
                ax.plot(s['round'], s[primary_metric], color=color, **PRIMARY_METRIC_STYLE[primary_metric])
                drawn_any = True
            if 'mia_auc' in s.columns and s['mia_auc'].notna().any():
                auc_rounds, auc_interp, auc_observed = prepare_auc_curve(s)
                ax2.plot(auc_rounds.values, auc_interp.values, color=color, **AUC_STYLE)
                observed_mask = auc_observed.notna()
                ax2.scatter(auc_rounds.values[observed_mask.values], auc_observed.values[observed_mask.values], color=color, s=18, alpha=0.9, zorder=4)
                drawn_any = True

        if not drawn_any:
            ax.text(0.5, 0.5, 'No plottable curves', ha='center', va='center', fontsize=13, color='gray', transform=ax.transAxes)

        ax.set_title(FRAMEWORK_TITLES[framework], fontsize=14, weight='bold')
        ax.set_xlabel('Round')
        ax.set_ylabel(primary_label, color='#222222')
        ax2.set_ylabel('MIA AUC', color='#444444')
        ax.grid(True, alpha=0.25)
        ax2.grid(False)
        ax.set_facecolor('#fbfbfc')
        ax.axhline(0, color='#d9d9d9', lw=0.8, alpha=0.4)
        ax.set_xlim(left=0)
        ax2.set_ylim(0.0, 1.05)

    fig.suptitle(title, fontsize=18, weight='bold', y=1.08)
    add_combined_legend(fig, primary_label)
    fig.tight_layout()
    return fig, axes, right_axes


In [ ]:
plot_metric_vs_auc(df=df, primary_metric='test_accuracy', primary_label='Test Accuracy', title='FL Performance vs Membership Inference Risk: Accuracy and MIA AUC')
plt.show()

In [ ]:
plot_metric_vs_auc(df=df, primary_metric='val_loss', primary_label='Validation Loss', title='FL Generalization vs Membership Inference Risk: Val Loss and MIA AUC')
plt.show()

# Baselines

这一节整理 `local`、`fedavg`、`fedprox`、`fedditto` 四类 baseline 结果。

汇总口径：
- `Accuracy (%)`：该次训练达到的最佳 test accuracy，换算成百分比
- `Converged Round`：验证损失最低时对应的 round
- `Train Time (s)`：日志最后一轮累计训练时间
- `Communication (MB)`：日志最后一轮累计通信量，按 MB 显示；`local` 记为 0


In [ ]:
def load_baseline_jsonl(path: Path) -> pd.DataFrame:
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    if not rows:
        return pd.DataFrame()

    config = rows[0]
    train_rows = [r for r in rows[1:] if 'round' in r]
    if not train_rows:
        return pd.DataFrame()

    df = pd.DataFrame(train_rows).sort_values('round').reset_index(drop=True)
    if 'global_test_accuracy' in df.columns and 'test_accuracy' not in df.columns:
        df = df.rename(columns={'global_test_accuracy': 'test_accuracy'})
    if 'personalized_val_loss' in df.columns and 'val_loss' not in df.columns:
        df = df.rename(columns={'personalized_val_loss': 'val_loss'})
    if 'global_test_loss' in df.columns and 'test_loss' not in df.columns:
        df = df.rename(columns={'global_test_loss': 'test_loss'})
    if 'communication_cost_bytes' not in df.columns:
        df['communication_cost_bytes'] = 0

    df['method'] = path.parent.name
    df['dataset'] = config.get('dataset')
    df['partition'] = config.get('partition')
    df['setting'] = f"{config.get('dataset')}_{config.get('partition')}"
    df['source_file'] = str(path)
    return df


def collect_baseline_results(result_dir: Path) -> pd.DataFrame:
    frames = []
    for method in BASELINE_METHODS:
        method_dir = result_dir / method
        if not method_dir.exists():
            continue
        for dataset, partition in SETTINGS:
            file_path = method_dir / f'{dataset}_{partition}.jsonl'
            if file_path.exists():
                frames.append(load_baseline_jsonl(file_path))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def collect_dp_mia_results(result_dir: Path) -> pd.DataFrame:
    import re

    frames = []
    pattern = re.compile(
        r'^(?P<dataset>mnist|cifar10)_(?P<partition>iid|shard)'
        r'_sigma\((?P<sigma>[^)]+)\)_q\((?P<q>[^)]+)\)_C\((?P<C>[^)]+)\)\.jsonl$'
    )

    for framework in FRAMEWORKS:
        framework_dir = result_dir / f'{framework}_dp_MIA'
        if not framework_dir.exists():
            continue
        for file_path in framework_dir.glob('*.jsonl'):
            match = pattern.match(file_path.name)
            if not match:
                continue
            frame = load_mia_jsonl(file_path)
            if frame.empty:
                continue
            frame['sigma'] = float(match.group('sigma'))
            frame['q'] = float(match.group('q'))
            frame['C'] = float(match.group('C'))
            frame['dp_run_label'] = f"sigma={match.group('sigma')}, q={match.group('q')}, C={match.group('C')}"
            frames.append(frame)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def select_dp_mia_run(df: pd.DataFrame, sigma: float, q: float, C: float) -> pd.DataFrame:
    if df.empty:
        return df
    return df[np.isclose(df['sigma'], sigma) & np.isclose(df['q'], q) & np.isclose(df['C'], C)].copy()


baseline_df = collect_baseline_results(RESULT_DIR)

In [ ]:
dp_mia_df = collect_dp_mia_results(RESULT_DIR)
dp_view = select_dp_mia_run(dp_mia_df, sigma=0.5, q=0.2, C=0.5)
plot_metric_vs_auc(dp_view, 'test_accuracy', 'Test Accuracy', 'DP-MIA: Accuracy and MIA AUC')
summary_dp_mia_df = dp_mia_df.groupby(['framework', 'dataset', 'partition', 'sigma', 'q', 'C']).agg({
    'test_accuracy': 'max',
    'mia_auc': 'max',
    # 'round': lambda s: int(s['round'].max()) if not s.empty else 0,
}).reset_index()
summary_dp_mia_df.rename(columns={'test_accuracy': 'max_test_accuracy', 'mia_auc': 'max_mia_auc'}, inplace=True)
summary_dp_mia_df
dp_mia_df

In [ ]:
"""
Accuracy (%)：该次训练达到的最佳 test accuracy
Converged Round：验证损失最低时对应的 round
Train Time (s)：最后一轮累计训练时间
Communication (MB)：最后一轮累计通信量，local 为 0
"""


def summarize_baselines(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    summary_rows = []
    for method in BASELINE_METHODS:
        for dataset, partition in SETTINGS:
            setting = f'{dataset}_{partition}'
            sub = df[(df['method'] == method) & (df['dataset'] == dataset) & (df['partition'] == partition)].copy()
            if sub.empty:
                summary_rows.append({
                    'method': BASELINE_TITLES[method],
                    'setting': setting,
                    'Accuracy (%)': np.nan,
                    'Converged Round': np.nan,
                    'Train Time (s)': np.nan,
                    'Communication (MB)': np.nan,
                })
                continue

            best_acc = sub['test_accuracy'].max() * 100.0
            best_val_idx = sub['val_loss'].idxmin()
            converged_round = int(sub.loc[best_val_idx, 'round'])
            final_train_time = float(sub['train_time'].iloc[-1])
            final_comm_mb = float(sub['communication_cost_bytes'].iloc[-1]) / (1024 ** 2)
            summary_rows.append({
                'method': BASELINE_TITLES[method],
                'setting': setting,
                'Accuracy (%)': round(best_acc, 2),
                'Converged Round': converged_round,
                'Train Time (s)': round(final_train_time, 2),
                'Communication (MB)': round(final_comm_mb, 2),
            })

    summary_df = pd.DataFrame(summary_rows)
    table = summary_df.pivot(index='method', columns='setting')
    ordered_columns = []
    for setting in ['mnist_iid', 'mnist_shard', 'cifar10_iid', 'cifar10_shard']:
        for metric in ['Accuracy (%)', 'Converged Round', 'Train Time (s)', 'Communication (MB)']:
            ordered_columns.append((metric, setting))
    table = table.reindex(index=[BASELINE_TITLES[m] for m in BASELINE_METHODS])
    table = table.reindex(columns=ordered_columns)
    table.columns = pd.MultiIndex.from_tuples(table.columns)
    return table, summary_df


baseline_summary, baseline_summary_df = summarize_baselines(baseline_df)
display(baseline_summary_df)

In [ ]:
def plot_baseline_run(df: pd.DataFrame, method: str, dataset: str, partition: str, figsize=(14, 5)):
    sub = df[(df['method'] == method) & (df['dataset'] == dataset) & (df['partition'] == partition)].copy()
    if sub.empty:
        raise ValueError(f'No baseline run found for {method} / {dataset}_{partition}')

    sub = sub.sort_values('round')
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    fig.patch.set_facecolor('white')

    color_loss = '#c44e52'
    color_acc = '#4c72b0'

    axes[0].plot(sub['round'], sub['val_loss'], color=color_loss, linewidth=2.4)
    best_idx = sub['val_loss'].idxmin()
    axes[0].scatter(sub.loc[best_idx, 'round'], sub.loc[best_idx, 'val_loss'], color=color_loss, s=40, zorder=3)
    axes[0].set_title('Validation Loss vs Round', fontsize=13, weight='bold')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Val Loss')
    axes[0].grid(True, alpha=0.25)
    axes[0].set_facecolor('#fbfbfc')

    axes[1].plot(sub['round'], sub['test_accuracy'] * 100.0, color=color_acc, linewidth=2.4)
    best_acc_idx = sub['test_accuracy'].idxmax()
    axes[1].scatter(sub.loc[best_acc_idx, 'round'], sub.loc[best_acc_idx, 'test_accuracy'] * 100.0, color=color_acc, s=40, zorder=3)
    axes[1].set_title('Test Accuracy vs Round', fontsize=13, weight='bold')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Test Accuracy (%)')
    axes[1].grid(True, alpha=0.25)
    axes[1].set_facecolor('#fbfbfc')

    title = f"{BASELINE_TITLES.get(method, method)} | {SETTING_LABELS[(dataset, partition)]}"
    fig.suptitle(title, fontsize=16, weight='bold', y=1.02)
    fig.tight_layout()
    return fig, axes


# Example:
for method in BASELINE_METHODS:
    for dataset, partition in SETTINGS:
        plot_baseline_run(baseline_df, method=method, dataset=dataset, partition=partition)
        plt.show()

## Notes

- 颜色编码数据集设定：同一种颜色对应同一个 `dataset-partition`
- 实线表示主指标（`Accuracy` 或 `Val Loss`）
- 虚线表示插值后的连续 `MIA AUC` 曲线，圆点表示实际有做 MIA 评估的轮次
- 左轴是主指标，右轴是 `MIA AUC`
- `shard` 在图中按 `NonIID` 标注
- baseline 汇总表中的 `Converged Round` 定义为验证损失达到最小值的轮次